In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import re 
import pickle
import pandas as pd
from torch.optim.lr_scheduler import StepLR 

In [3]:
hidden_size = 256
PAD_taken =0
SOS_token = 1
EOS_token = 2
UNK_token = 4
MAX_LENGTH = 300
device = torch.device("mps")

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = text.lower()
    text = re.sub(r'\d+',' ', text)
    text = re.sub(r'([^\w\s])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

In [7]:
def indexesFromSentence(vocab, sentence):
    return [vocab.get(word, vocab['<UNK>']) for word in sentence.split(' ')]

def tensorFromSentence(vocab, sentence):
    indexes = indexesFromSentence(vocab, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype = torch.long, device =device).view(-1,1)

class EncoderLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=2)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1,1,-1)
        output,hidden = self.lstm(embedded, hidden)
        return output, hidden
    def initHidden(self):
        return(torch.zeros(2,1,self.hidden_size, device= device),
               torch.zeros(2,1,self.hidden_size, device= device))

In [11]:
class AttnDecoderLSTM(nn.Module):
    def __init__(self,hidden, output_size):
        super(AttnDecoderLSTM, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size , self.hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers = 2)
        self.out = nn.Linear(self.hidden_size, output_size)

    def forward(self, input,hidden, encoder_outputs):
        embedded = self.embedding(input).view(1,1,-1)
        attn_weights = F.softmax(torch.bmm(encoder_outputs.unsqueeze(0), hidden[0][0].unsqueeze(2)).squeeze(2), dim = 1)
        attn_applied= torch.bmm(attn_weights.unsqueeze(0), encoder_outputs.unsqueeze(0))

        new_hidden = (torch.vstack([attn_applied, attn_applied]), hidden[1])
        output,hidden = self.lstm(embedded[0].unsqueeze(0), new_hidden)
        output = self.out(output[0])
        return output, hidden, attn_weights
    
    def initHidden(self):
        return (torch.zeros(2,1,self.hidden_size, device=device),
                torch.zeros(2,1,self.hidden_size,device=device))

In [12]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length = MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_gard()
    decoder_optimizer.zero_gard()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
    loss = 0

    for ei in range(input_length):
        encoder_outputs, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] += encoder_output[0,0]
    
    decoder_input = torch.tensor([[SOS_token]],device= device)
    decoder_hidden = encoder_hidden
    
    for di in range(target_length):
        decoder_output, decoder_hidden,decoder_attention  = decoder(decoder_input, decoder_hidden, encoder_outputs)
        topv, topi = decoder_output.topk(1)
        decoder_input = topi.squeeze().detach()
        loss += criterion(decoder_output, target_tensor[di])
        if decoder_input.item() == EOS_token:
            break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [13]:
def trainIters(encoder, decoder, n_iters, print_every = 1000):
    print_loss_total = 0

    scheduler_encoder = StepLR(encoder_optimizer, step_size = 10, gamma = 0.1)
    scheduler_decoder = StepLR(decoder_optimizer, step_size=10, gamma= 0.1)
    min_loss = 1000000
    for iter in range(1, n_iters+1):
        training_pair = random.choice(pairs)
        input_tensor = tensorFromSentence(word_to_ix, training_pair[0]).to(device)
        target_tensor = tensorFromSentence(word_to_ix, training_pair[1]).to(device)

        loss = train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss

        if iter %  print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print(f'Iteration: {iter}, Losss: {print_loss_avg: .4f}, enc_lr: {scheduler_encoder.get_lr()}')

            print_loss_total =0

            if min_loss > print_loss_avg:
                torch.save(encoder.state_dict(), '.models/chkpt/encoder_seq2seq_attention_dot_' + str(iter) + '.pth')
                torch.save(decoder.state_dict(), '.models/chkpt/decoder_seq2seq_attention_dot_' + str(iter) + '.pth')
                min_loss = print_loss_avg

            scheduler_encoder.step()
            scheduler_decoder.step()
            

In [ ]:
def evaluate(encoder, decoder, sentence, max_length = MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(word_to_ix, sentence).to(device)
        input_length = input_tensor.size(0)
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            